In [ ]:
!pip install ultralytics kagglehub opencv-python

  Using cached ultralytics-8.4.24-py3-none-any.whl.metadata (39 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
Using cached ultralytics-8.4.24-py3-none-any.whl (1.2 MB)
Using cached ultralytics_thop-2.0.18-py3-none-any.whl (28 kB)


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("swish9/weeds-detection")

print("Path to dataset files:", path)

100%|██████████| 106M/106M [00:03<00:00, 31.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/swish9/weeds-detection/versions/1


In [ ]:
import os
import shutil
from collections import Counter

dataset_path = "/content/drive/MyDrive/rover-weed/dataset/dataset"
print("Fixed Dataset Analysis:\n")

def safe_analyze_dataset(root_path):
    total_imgs, total_lbls = 0, 0

    for root, dirs, files in os.walk(root_path):
        num_files = len(files)
        if num_files > 0:
            print(f"{root}: {num_files} files")
            img_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            lbl_files = [f for f in files if f.endswith('.txt')]
            print(f"  Images: {len(img_files)}, Labels: {len(lbl_files)}")

            # Safe label parsing
            classes = []
            for lbl in lbl_files[:3]:
                lbl_path = os.path.join(root, lbl)
                try:
                    with open(lbl_path, 'r') as f:
                        content = f.read().strip()
                        print(f"  Sample label content: {content[:100]}...")

                        # Try numeric first, then named classes
                        for line in content.split('\n'):
                            if line.strip():
                                parts = line.strip().split()
                                if parts[0].isdigit():
                                    classes.append(int(parts[0]))
                                elif parts[0] in ['crop', 'weed']:
                                    classes.append(0 if parts[0] == 'crop' else 1)
                except:
                    print(f"  Could not parse {lbl}")

            if classes:
                print(f"  Classes found: {dict(Counter(classes))}")
            print()

        total_imgs += len(img_files)
        total_lbls += len(lbl_files)

    print(f"TOTAL: {total_imgs} images, {total_lbls} labels")

safe_analyze_dataset(dataset_path)


Fixed Dataset Analysis:

/content/drive/MyDrive/rover-weed/dataset/dataset: 1 files
  Images: 0, Labels: 1
  Sample label content: crop
weed...
  Classes found: {0: 1, 1: 1}

/content/drive/MyDrive/rover-weed/dataset/dataset/labels/test: 244 files
  Images: 0, Labels: 244
  Sample label content: 0 0.529297 0.437500 0.855469 0.605469...
  Sample label content: 1 0.508789 0.489258 0.869141 0.861328...
  Sample label content: 0 0.532227 0.537109 0.853516 0.902344...
  Classes found: {0: 2, 1: 1}

/content/drive/MyDrive/rover-weed/dataset/dataset/labels/val: 247 files
  Images: 0, Labels: 247
  Sample label content: 1 0.397461 0.558594 0.693359 0.437500...
  Sample label content: 0 0.770508 0.640625 0.087891 0.203125
0 0.482422 0.217773 0.078125 0.185547
0 0.491211 0.763672 0.16...
  Sample label content: 1 0.507812 0.500977 0.171875 0.169922
1 0.607422 0.097656 0.128906 0.191406...
  Classes found: {1: 3, 0: 7}

/content/drive/MyDrive/rover-weed/dataset/dataset/labels/train: 1245 files
  

In [ ]:
import os

dataset_root = "/content/drive/MyDrive/rover-weed/dataset/dataset"
yaml_path = "/content/weed_only_correct.yaml"

yaml_content = f"""path: {dataset_root}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['weed']
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("✅ CORRECT YAML created for your structure")
print(f"Dataset root: {dataset_root}")


✅ CORRECT YAML created for your structure
Dataset root: /content/drive/MyDrive/rover-weed/dataset/dataset


In [ ]:
# Test paths
print("FINAL PATH CHECK:")
print(f"Train imgs: {os.path.exists('/content/drive/MyDrive/rover-weed/dataset/dataset/images/train')}")
print(f"Val imgs:   {os.path.exists('/content/drive/MyDrive/rover-weed/dataset/dataset/images/val')}")
print(f"Train lbls: {os.path.exists('/content/drive/MyDrive/rover-weed/dataset/dataset/labels/train')}")
print(f"Val lbls:   {os.path.exists('/content/drive/MyDrive/rover-weed/dataset/dataset/labels/val')}")


FINAL PATH CHECK:
Train imgs: True
Val imgs:   True
Train lbls: True
Val lbls:   True


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/weed_only_correct.yaml',  # Local YAML
    epochs=100,
    imgsz=640,
    batch=16,
    name='rover-weed-perfect',
    project='/content/drive/MyDrive/rover-weed/runs',
    patience=15,
    save_period=10
)


Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/weed_only_correct.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rover-weed-perfect2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15